#### Creating Dataframe object for the jobs that need to be scheduled

In [14]:
import pandas as pd

# Storing all job data details
jobs_data = {
    "jobs": ["A", "B", "C", "D", "E"],
    "arrival_time": [0, 1, 2, 3, 4],
    "processing_time": [11, 29, 31, 1, 2],
    "due_date": [61, 45, 31, 33, 32]
}

# Convert the raw job data into a DataFrame for scheduling.
df = pd.DataFrame(jobs_data)
df

,jobs,arrival_time,processing_time,due_date
0,A,0,11,61
1,B,1,29,45
2,C,2,31,31
3,D,3,1,33
4,E,4,2,32


#### Earliest Due Date Dynamic Job Scheduling

In [15]:
# Sort by arrival time so the dynamic queue can be evaluated chronologically.
df_dynamic = df.copy().sort_values(by='arrival_time')
df_dynamic

,jobs,arrival_time,processing_time,due_date
0,A,0,11,61
1,B,1,29,45
2,C,2,31,31
3,D,3,1,33
4,E,4,2,32


In [16]:
# Track which jobs are complete and store their performance metrics.
df_dynamic["is_processed"] = False
df_dynamic["finish_time"] = None
df_dynamic["flow_time"] = None
df_dynamic["tardiness"] = None

current_time = 0

# Iterating over the jobs till not all jobs are processed
while not df_dynamic["is_processed"].all():

    # Checking all the available jobs
    available_jobs = df_dynamic[(df_dynamic["is_processed"] == False) & (df_dynamic["arrival_time"] <= current_time)]

    # If no job is available, move time to the next job arrival
    if available_jobs.empty:
        next_arrival_time = df_dynamic.loc[df_dynamic["is_processed"] == False, "arrival_time"].min()
        current_time = next_arrival_time
        continue

    # Finding job with the earliest due date
    earliest_due_date_index = available_jobs["due_date"].idxmin()

    # Updating finish time
    finish_time = current_time + df_dynamic.loc[earliest_due_date_index, "processing_time"]

    # Updating data in the dataframe for the calculated values
    df_dynamic.loc[earliest_due_date_index, "is_processed"] = True
    df_dynamic.loc[earliest_due_date_index, "finish_time"] = finish_time
    df_dynamic.loc[earliest_due_date_index, "flow_time"] = finish_time - df_dynamic.loc[earliest_due_date_index, "arrival_time"]
    df_dynamic.loc[earliest_due_date_index, "tardiness"] = max(finish_time - df_dynamic.loc[earliest_due_date_index, "due_date"], 0)
    
    # Updating current time with the finish time of the current job
    current_time = finish_time

In [17]:
df_dynamic

,jobs,arrival_time,processing_time,due_date,is_processed,finish_time,flow_time,tardiness
0,A,0,11,61,True,11,11,0
1,B,1,29,45,True,74,73,29
2,C,2,31,31,True,42,40,11
3,D,3,1,33,True,45,42,12
4,E,4,2,32,True,44,40,12


In [18]:
avg_tardiness = df_dynamic['tardiness'].mean()
avg_flow_time = df_dynamic['flow_time'].mean()
print(f"Avg Tardiness: {avg_tardiness}, Avg Flow time: {avg_flow_time}")

Avg Tardiness: 12.8, Avg Flow time: 41.2


#### Earliest Due Date Static Job Scheduling

In [23]:
# Creating copy of the original dataframe
df_static = df.copy()

# Sort by due date so that the job can be evaluated as static scheduling
df_static = df_static.sort_values(by='due_date')
df_static

,jobs,arrival_time,processing_time,due_date
2,C,2,31,31
4,E,4,2,32
3,D,3,1,33
1,B,1,29,45
0,A,0,11,61


In [24]:
# Is Processed, Finish Time, Flow Time(Finish Time - Arrival Time), Tardiness(max(Finish Time - Due Date, 0))
df_static["finish_time"] = None
df_static["flow_time"] = None
df_static["tardiness"] = None

current_time = 0

# Iterating over the received jobs
for index, row in df_static.iterrows():
    finish_time = current_time + row.processing_time

    # Updating data in the dataframe for the calculated values
    df_static.loc[index, "finish_time"] = finish_time
    df_static.loc[index, "flow_time"] = finish_time - row.arrival_time
    df_static.loc[index, "tardiness"] = max(finish_time - row.due_date, 0)

    # Updating current time with the finish time of the current job
    current_time = finish_time

    # DEBUG LOG: Uncomment this line to see logs
    # print(f"Index: {index}, Job: {row.jobs}, Arrival Time: {row.arrival_time}, Finish time: {finish_time}, Flow time: {finish_time - row.arrival_time}")

In [25]:
df_static

,jobs,arrival_time,processing_time,due_date,finish_time,flow_time,tardiness
2,C,2,31,31,31,29,0
4,E,4,2,32,33,29,1
3,D,3,1,33,34,31,1
1,B,1,29,45,63,62,18
0,A,0,11,61,74,74,13


In [26]:
avg_tardiness = df_static['tardiness'].mean()
avg_flow_time = df_static['flow_time'].mean()
print(f"Avg Tardiness: {avg_tardiness}, Avg Flow time: {avg_flow_time}")

Avg Tardiness: 6.6, Avg Flow time: 45.0
